In [104]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split    # Train/test split
from sklearn.pipeline import Pipeline                  # Pipeline to prevent leakage
from sklearn.compose import ColumnTransformer        # Different preprocessing per column type
from sklearn.preprocessing import  RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression  , Lasso 
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.svm import SVR


In [85]:
df=pd.read_csv(r"data\ priceinr.csv", sep=(","))
df

,Unnamed: 0,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Power,Price,brand,model,Engine
0,0,2010,72000,CNG,Manual,First,26.60,26.60,1.75,MARUTI,WAGON R,998.0
1,1,2015,41000,Diesel,Manual,First,19.67,19.67,12.50,HYUNDAI,CRETA 1.6,1582.0
2,2,2011,46000,Petrol,Manual,First,18.20,18.20,4.50,HONDA,JAZZ V,1199.0
3,3,2012,87000,Diesel,Manual,First,20.77,20.77,6.00,MARUTI,ERTIGA VDI,1248.0
4,4,2013,40670,Diesel,Automatic,Second,15.20,15.20,17.74,AUDI,A4 NEW,1968.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5978,6014,2014,27365,Diesel,Manual,First,28.40,28.40,4.75,MARUTI,SWIFT VDI,1248.0
5979,6015,2015,100000,Diesel,Manual,First,24.40,24.40,4.00,HYUNDAI,XCENT 1.1,1120.0
5980,6016,2012,55000,Diesel,Manual,Second,14.00,14.00,2.90,MAHINDRA,XYLO D4,2498.0
5981,6017,2013,46000,Petrol,Manual,First,18.90,18.90,2.65,MARUTI,WAGON R,998.0


In [86]:
df.Power.isnull().sum()

np.int64(2)

In [87]:

df["Power"] = df["Power"].replace(["null", "NaN", "-", "unknown"], np.nan)

In [88]:
df.dropna(subset=["Power"])

,Unnamed: 0,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Power,Price,brand,model,Engine
0,0,2010,72000,CNG,Manual,First,26.60,26.60,1.75,MARUTI,WAGON R,998.0
1,1,2015,41000,Diesel,Manual,First,19.67,19.67,12.50,HYUNDAI,CRETA 1.6,1582.0
2,2,2011,46000,Petrol,Manual,First,18.20,18.20,4.50,HONDA,JAZZ V,1199.0
3,3,2012,87000,Diesel,Manual,First,20.77,20.77,6.00,MARUTI,ERTIGA VDI,1248.0
4,4,2013,40670,Diesel,Automatic,Second,15.20,15.20,17.74,AUDI,A4 NEW,1968.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5978,6014,2014,27365,Diesel,Manual,First,28.40,28.40,4.75,MARUTI,SWIFT VDI,1248.0
5979,6015,2015,100000,Diesel,Manual,First,24.40,24.40,4.00,HYUNDAI,XCENT 1.1,1120.0
5980,6016,2012,55000,Diesel,Manual,Second,14.00,14.00,2.90,MAHINDRA,XYLO D4,2498.0
5981,6017,2013,46000,Petrol,Manual,First,18.90,18.90,2.65,MARUTI,WAGON R,998.0


In [89]:
df.dropna(subset=["Power"], inplace=True)


In [90]:
df.Power.isna().sum()

np.int64(0)

In [91]:
df["brand"].value_counts()

brand
MARUTI           1200
HYUNDAI          1100
HONDA             602
TOYOTA            409
MERCEDES-BENZ     318
VOLKSWAGEN        315
FORD              300
MAHINDRA          271
BMW               262
AUDI              236
TATA              186
SKODA             173
RENAULT           145
CHEVROLET         121
NISSAN             91
LAND               57
JAGUAR             40
MITSUBISHI         27
MINI               26
FIAT               25
VOLVO              21
PORSCHE            18
JEEP               15
DATSUN             13
ISUZU               3
FORCE               3
SMART               1
AMBASSADOR          1
BENTLEY             1
LAMBORGHINI         1
Name: count, dtype: int64

In [92]:
df.info()

<class 'pandas.DataFrame'>
Index: 5981 entries, 0 to 5982
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5981 non-null   int64  
 1   Year               5981 non-null   int64  
 2   Kilometers_Driven  5981 non-null   int64  
 3   Fuel_Type          5981 non-null   str    
 4   Transmission       5981 non-null   str    
 5   Owner_Type         5981 non-null   str    
 6   Mileage            5981 non-null   float64
 7   Power              5981 non-null   float64
 8   Price              5981 non-null   float64
 9   brand              5981 non-null   str    
 10  model              5981 non-null   str    
 11  Engine             5981 non-null   float64
dtypes: float64(4), int64(3), str(5)
memory usage: 607.4 KB


In [93]:
x=df.drop(columns=["Price"])
y=df["Price"]


In [94]:
numeric_features = x.select_dtypes(include=[np.number]).columns.tolist()   # Numeric columns list.tolist()  
categorical_features = x.select_dtypes(exclude=[np.number]).columns.tolist()

In [95]:
numeric_features

['Unnamed: 0', 'Year', 'Kilometers_Driven', 'Mileage', 'Power', 'Engine ']

In [96]:
categorical_features

['Fuel_Type', 'Transmission', 'Owner_Type', 'brand', 'model']

In [97]:
#convertit des données nominales aux valeurs numériques 1 2 3 4 5
# df["Fuel_Type"]=pd.factorize(df["Fuel_Type"])[0]
# df["Transmission"]=pd.factorize(df["Transmission"])[0]
# df["Owner_Type"]=pd.factorize(df["Owner_Type"])[0]
# df["Brand"]=pd.factorize(df["Brand"])[0]
# df["Model"]=pd.factorize(df["Model"])[0]
# df
#df = pd.get_dummies(df, columns=["Transmission", "Owner_Type", "Model","Brand"], dtype=float)

In [98]:
# df= df.drop(["Fuel_Type"], axis=1)
# df

In [99]:

# Séparer x et y
x = df.drop(columns=["Price"])
y = df["Price"]

# Colonnes
numeric_features = x.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = x.select_dtypes(exclude=[np.number]).columns.tolist()

# Transformations
numeric_transformer = Pipeline(steps=[
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("numerique", numeric_transformer, numeric_features),
        ("categorial", categorical_transformer, categorical_features)
    ]
)

# Pipeline complet
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

# Split
x_train, x_test, y_train, y_test = train_test_split( x, y, test_size=0.2, random_state=42)
print(x_train.shape , x_test.shape) 
# Entraînement
model.fit(x_train, y_train)


(4784, 11) (1197, 11)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerique', ...), ('categorial', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [100]:
y_pred = model.predict(x_test)
mse=mean_squared_error(y_test, y_pred)
rmse=root_mean_squared_error(y_test , y_pred)
mae=mean_absolute_error(y_test, y_pred)
r2=r2_score(y_test , y_pred)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.4f}")

MSE  : 18.25
RMSE : 4.27
MAE  : 2.31
R²   : 0.8491


c:\Users\samya.taqi\Desktop\SAMYA TAQI\Pr-dire-le-prix-de-voiture--projet-r-gression\price_car\pricecarenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [102]:

# Séparer x et y
x = df.drop(columns=["Price"])
y = df["Price"]

# Colonnes
numeric_features = x.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = x.select_dtypes(exclude=[np.number]).columns.tolist()

# Transformations
numeric_transformer = Pipeline(steps=[
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("numerique", numeric_transformer, numeric_features),
        ("categorial", categorical_transformer, categorical_features)
    ]
)

# Pipeline complet
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", Lasso(alpha=0.1))
])

# Split
x_train, x_test, y_train, y_test = train_test_split( x, y, test_size=0.2, random_state=42)
print(x_train.shape , x_test.shape) 
# Entraînement
model.fit(x_train, y_train)

(4784, 11) (1197, 11)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerique', ...), ('categorial', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [103]:
# Faire des prédictions
y_pred = model.predict(x_test)

# Évaluer
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE = {mse:.2f}")
print(f"R²  = {r2:.2f}")

MSE = 38.81
R²  = 0.68


c:\Users\samya.taqi\Desktop\SAMYA TAQI\Pr-dire-le-prix-de-voiture--projet-r-gression\price_car\pricecarenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [110]:
# Séparer x et y
x = df.drop(columns=["Price"])
y = df["Price"]

# Colonnes
numeric_features = x.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = x.select_dtypes(exclude=[np.number]).columns.tolist()

# Transformations
numeric_transformer = Pipeline(steps=[
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ("numerique", numeric_transformer, numeric_features),
        ("categorial", categorical_transformer, categorical_features)
    ]
)


model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("svm", SVR(kernel="rbf", C=1.0, epsilon=0.1))
])

# Split
x_train, x_test, y_train, y_test = train_test_split( x, y, test_size=0.2, random_state=42)
print(x_train.shape , x_test.shape) 
# Entraînement
model.fit(x_train, y_train)

(4784, 11) (1197, 11)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerique', ...), ('categorial', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [111]:
y_pred=model.predict(x_test)
# Évaluer
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE = {mse:.2f}")
print(f"R²  = {r2:.2f}")

MSE = 29.15
R²  = 0.76


c:\Users\samya.taqi\Desktop\SAMYA TAQI\Pr-dire-le-prix-de-voiture--projet-r-gression\price_car\pricecarenv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
